In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, WARNING, ERROR, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import

In [ ]:
import copy
import math
import os
import pathlib
from itertools import product
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

from src.simulation.seasonal_sro import (
    SeasonalSROConfig,
    simulate_SeasonalSRO_and_write_out_results,
)
from src.utils.random_seed_helper import set_all_seeds

# Define constants

In [ ]:
ROOT_DIR = str((pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]) / "..").resolve())
DATA_KIND = "KA21"  # "KA21", "V25", "H26"
assert DATA_KIND in ["KA21", "V25", "H26"]

In [ ]:
DEFAULT = SeasonalSROConfig(value_set_name=DATA_KIND)

In [ ]:
RESULT_DIR = f"{ROOT_DIR}/data/models"
os.makedirs(RESULT_DIR, exist_ok=True)

# Dependence on Ra

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(1)
multiprocessing.set_start_method("spawn", force=True)
max_workers = 26 if int(os.cpu_count()) > 26 else os.cpu_count()
if DATA_KIND == "H26":
    max_workers = 61 if int(os.cpu_count()) > 61 else os.cpu_count()

print(f"{max_workers=}")

In [ ]:
lst_amp_Ra = np.linspace(0, 2.5, 26)

if DATA_KIND == "H26":
    lst_amp_Ra = np.linspace(0, 6.0, 61)

print(f"{lst_amp_Ra=}")

# Set t_max = 50_000 to reproduce our results.
for t_max, is_random_init in product([10_000], [False]):

    out_dir = f"{ROOT_DIR}/data/models/SRO_long_run_{t_max:06}yr_{DATA_KIND}"
    if is_random_init:
        out_dir += "_random_init"
    os.makedirs(out_dir, exist_ok=True)
    print(f"Calculating: {t_max=}, {is_random_init=}")

    base_seed = 42
    n_batches = 1
    t_min = 0
    n_per_month = 30  # days per month
    n_timesteps = int((t_max - t_min) * 12 * n_per_month)

    set_all_seeds(base_seed)
    x0 = torch.tensor([0.0, 10.0], dtype=torch.float64).unsqueeze(0)  # shape (1, 2)
    x0 = x0.repeat(n_batches, 1)  # shape (n_batches, 2)

    if is_random_init:
        print("Using random initial conditions.")
        x0 = torch.randn_like(x0)  # use this if you want random initial conditions

    tasks = list(enumerate(lst_amp_Ra))  # (index, amp_Ra)

    results = []
    with ProcessPoolExecutor(
        max_workers=max_workers,
    ) as ex:
        futures = []

        for index, amp in tasks:
            out_file_path = f"{out_dir}/SRO_Ra{str(amp).replace('.', 'p')[:3]}.pickle"

            fut = ex.submit(
                simulate_SeasonalSRO_and_write_out_results,
                amp,
                base_seed,
                copy.deepcopy(DEFAULT),
                x0,
                t_min,
                t_max,
                n_timesteps,
                n_batches,
                out_file_path,
                "float32",
                index,
            )
            futures.append(fut)

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Simulating"):
            try:
                res = fut.result()
                assert res["status"] == "ok" or res["status"] == "skipped-exists"
            except Exception as e:
                res = {"ERROR": repr(e)}
            results.append(res)

# Dependence on Ra and intrinsic frequency

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
torch.set_num_threads(1)
multiprocessing.set_start_method("spawn", force=True)
max_workers = 140 if int(os.cpu_count()) > 140 else os.cpu_count()
print(f"{max_workers=}")

In [ ]:
lst_omega = np.linspace(0.5, 2.5, 21)
lst_amp_Ra = np.linspace(0, 2.5, 26)

# Set t_max = 50_000 to reproduce our results.
for t_max, is_random_init in product([10_000], [False]):
    out_dir = (
        f"{ROOT_DIR}/data/models/omega_vs_Ra_SRO_long_run_{t_max:06}yr_{DATA_KIND}"
    )
    if is_random_init:
        out_dir += "_random_init"
    os.makedirs(out_dir, exist_ok=True)

    base_seed = 42
    n_batches = 1
    t_min = 0
    n_per_month = 30  # days per month
    n_timesteps = int((t_max - t_min) * 12 * n_per_month)

    set_all_seeds(base_seed)
    x0 = torch.tensor([0.0, 10.0], dtype=torch.float64).unsqueeze(0)  # shape (1, 2)
    x0 = x0.repeat(n_batches, 1)  # shape (n_batches, 2)
    if is_random_init:
        print("Using random initial conditions.")
        x0 = torch.randn_like(x0)  # use this if you want random initial conditions

    tasks = list(enumerate(product(lst_omega, lst_amp_Ra)))  # (index, (omega, amp_Ra))

    results = []
    with ProcessPoolExecutor(
        max_workers=max_workers,
    ) as ex:
        futures = []

        for index, (omega, amp) in tasks:
            _f = str(omega).replace(".", "p")[:3]
            _r = str(amp).replace(".", "p")[:3]
            out_file_path = f"{out_dir}/SRO_Frq{_f}_Ra{_r}.pickle"

            cfg = copy.deepcopy(DEFAULT)
            cfg.F1 = math.sqrt(omega**2 + cfg.R0**2 / 4.0)
            cfg.F2 = cfg.F1
            cfg.epsilon = 0.0
            cfg.Ra = abs(cfg.R0) * amp
            assert abs(cfg.get_bwj_imaginary_part() - omega) < 1e-7

            fut = ex.submit(
                simulate_SeasonalSRO_and_write_out_results,
                amp,
                base_seed,
                cfg,
                x0,
                t_min,
                t_max,
                n_timesteps,
                n_batches,
                out_file_path,
                "float32",
                index,
            )
            futures.append(fut)

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Simulating"):
            try:
                res = fut.result()
                assert res["status"] == "ok" or res["status"] == "skipped-exists"
            except Exception as e:
                res = {"status": "error", "error": repr(e)}
            results.append(res)